# NASA Turbofan Motorları — Canlı Model Demosu (Gradio)

Bu defter, eğittiğimiz Random Forest modelini **gerçekten çalıştıran** bir arayüz açar. Web sayfası demosundan farkı: tahminler önceden hesaplanmış değil, **sen bir değeri değiştirdiğinde model o anda tahmin üretir.**

**Ne yapacaksın?**
1. Soldaki **📁 Dosyalar** paneline seçtiğin veri setinin 3 dosyasını sürükle: `train_FDxxx.txt`, `test_FDxxx.txt`, `RUL_FDxxx.txt`.
2. Aşağıdaki hücreleri sırayla çalıştır (her hücrenin altındaki ▶ düğmesi veya `Shift + Enter`).
3. Son hücre bir **public link** verir. Linki tarayıcıda aç, arayüzü kullan.

Kullanacağımız yöntem, ana defterlerdekiyle aynı: sabit sütunları at, RUL'u 125'te sınırla, `RandomForestRegressor(n_estimators=100, max_depth=10)` ile eğit.

### 1. Ayarlar
Hangi veri setiyle çalışacağını yaz. Dosyaları Colab'a yüklediysen `DATA_DIR` değişmez.

In [ ]:
DS = "FD001"           # FD001, FD002, FD003 veya FD004
DATA_DIR = "/content"   # Colab'da yüklediğin dosyaların bulunduğu yer

### 2. Gradio'yu kur
Colab'da Gradio çoğu zaman hazır gelir. Bu hücre emin olmak içindir.

In [ ]:
!pip install -q gradio

### 3. Veriyi hazırla ve modeli eğit
Ana defterdeki adımların kısa hali: dosyaları oku, değişmeyen sütunları at, `RUL = son çevrim − çevrim` hesapla, RUL'u 125'te sınırla, modeli eğit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

kolonlar = ["unit", "cycle"] + [f"setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]
df_train = pd.read_csv(f"{DATA_DIR}/train_{DS}.txt", sep=r"\s+", header=None, names=kolonlar)
df_test = pd.read_csv(f"{DATA_DIR}/test_{DS}.txt", sep=r"\s+", header=None, names=kolonlar)
df_rul = pd.read_csv(f"{DATA_DIR}/RUL_{DS}.txt", header=None, names=["RUL"])

# Bilgi vermeyen (neredeyse sabit) sütunları at
std = df_train.std()
sabit = std[std < 0.01].index
df_train = df_train.drop(columns=sabit)
df_test = df_test.drop(columns=sabit)

# RUL = motorun son çevrimi - şu anki çevrim, sonra 125'te sınırla
df_train["RUL"] = df_train.groupby("unit")["cycle"].transform("max") - df_train["cycle"]
features = df_train.columns.drop(["unit", "cycle", "RUL"])

X_train = df_train[features]
y_train = df_train["RUL"].clip(upper=125)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("Model eğitildi.", DS, "| özellik sayısı:", len(features), "| test motoru:", df_test["unit"].nunique())

### 4. Tahmin fonksiyonları
Arayüz bu iki fonksiyonu çağıracak.

- `motor_tahmini`: bir **test motoru** ve **çevrim** seçilir. Model, o çevrime kadarki her satır için RUL tahmin eder ve gerçek RUL ile çizer. Eşiğin altına inince bakım uyarısı verir.
- `sensor_tahmini`: sensör değerlerini **elle girersin**, model tek bir satıra bakarak RUL tahmin eder.

In [ ]:
def motor_tahmini(motor_no, cevrim, esik):
    motor_no = int(motor_no)
    motor = df_test[df_test["unit"] == motor_no].reset_index(drop=True)
    n = len(motor)
    cevrim = int(min(max(cevrim, 1), n))

    # Modelin tahmini: seçilen çevrime kadar her satır için
    tahminler = rf_model.predict(motor.iloc[:cevrim][features])
    son_tahmin = float(tahminler[-1])

    # Gerçek RUL: RUL dosyası motorun SON çevrimindeki değeri verir, geriye doğru sayarız
    son_gercek = int(df_rul["RUL"].iloc[motor_no - 1])
    gercek = np.minimum(125, son_gercek + (n - np.arange(1, cevrim + 1)))
    simdiki_gercek = int(gercek[-1])

    alarm = son_tahmin <= esik
    yakin = simdiki_gercek <= 30
    if alarm and yakin:
        durum = "Doğru alarm"
    elif alarm and not yakin:
        durum = "Yanlış alarm"
    elif not alarm and yakin:
        durum = "Kaçırılan arıza"
    else:
        durum = "Doğru: alarm yok"

    fig, ax = plt.subplots(figsize=(7, 3.6))
    x = np.arange(1, cevrim + 1)
    ax.plot(x, tahminler, label="Tahmin (Random Forest)", color="tab:blue")
    ax.plot(x, gercek, label="Gerçek RUL (en çok 125)", color="gray", linestyle="--")
    ax.axhline(esik, color="tab:orange", linewidth=1, label="Alarm eşiği")
    ax.set_xlim(1, n)
    ax.set_xlabel("Çevrim")
    ax.set_ylabel("Kalan ömür (çevrim)")
    ax.legend()
    plt.close(fig)

    yazi = (f"**Tahmini kalan ömür:** {son_tahmin:.0f} çevrim  \n"
            f"**Gerçek kalan ömür (en çok 125):** {simdiki_gercek} çevrim  \n"
            f"**Bakım uyarısı:** {'BAKIM PLANLA' if alarm else 'Alarm yok'}  \n"
            f"**Gerçekle karşılaştırma:** {durum}")
    return fig, yazi


def sensor_tahmini(*degerler):
    satir = pd.DataFrame([list(degerler)], columns=features)
    tahmin = float(rf_model.predict(satir)[0])
    return f"**Tahmini kalan ömür:** {tahmin:.0f} çevrim"

### 5. Fonksiyonları arayüz olmadan dene
Önce fonksiyon çalışıyor mu bakalım. 1 numaralı motorun son çevrimi için tahmini yazdır.

In [ ]:
ornek_fig, ornek_yazi = motor_tahmini(1, 9999, 40)
print(ornek_yazi)

### 6. Arayüzü aç
Bu hücreyi çalıştırınca aşağıda bir arayüz ve bir **public link** (`https://....gradio.live`) çıkar. Linki başka bir cihazda da açabilirsin. Link yaklaşık 1 hafta geçerlidir ve defter çalışırken açıktır; sunumdan hemen önce hücreyi tekrar çalıştır.

**İki sekme var:**
- **Test motoru:** motor, çevrim ve eşik kaydırıcılarını oynat; her değişiklikte model yeniden tahmin yapar.
- **Sensör değerlerini gir:** her özelliği kaydırıcıyla ayarla; model o satır için anında RUL tahmin eder.

In [ ]:
import gradio as gr

motor_sayisi = int(df_test["unit"].nunique())
ilk_n = int((df_test["unit"] == 1).sum())
ozet = df_train[features].agg(["min", "max", "median"])

with gr.Blocks(title="Motor Ömrü Tahmini") as demo:
    gr.Markdown("# Motor Ömrü Tahmini\nEğitilmiş Random Forest modeli, kaydırıcıları her oynattığında **canlı** tahmin üretir.")

    with gr.Tab("Test motoru"):
        with gr.Row():
            motor_in = gr.Slider(1, motor_sayisi, value=1, step=1, label="Motor numarası")
            cevrim_in = gr.Slider(1, ilk_n, value=ilk_n, step=1, label="Şu anki çevrim")
            esik_in = gr.Slider(20, 60, value=40, step=5, label="Bakım uyarı eşiği (çevrim)")
        grafik = gr.Plot(label="Tahmin ve gerçek RUL")
        yazi = gr.Markdown()

        def motor_degisti(motor_no):
            n = int((df_test["unit"] == int(motor_no)).sum())
            return gr.update(maximum=n, value=n)

        motor_in.change(motor_degisti, motor_in, cevrim_in)
        for girdi in (motor_in, cevrim_in, esik_in):
            girdi.change(motor_tahmini, [motor_in, cevrim_in, esik_in], [grafik, yazi])
        demo.load(motor_tahmini, [motor_in, cevrim_in, esik_in], [grafik, yazi])

    with gr.Tab("Sensör değerlerini gir"):
        gr.Markdown("Her özelliği ayarla. Başlangıç değeri eğitim verisinin **medyanıdır**. Model yalnızca bu tek satıra bakar, motorun geçmişini bilmez.")
        kaydiricilar = []
        for f in features:
            lo, hi, med = float(ozet.loc["min", f]), float(ozet.loc["max", f]), float(ozet.loc["median", f])
            kaydiricilar.append(gr.Slider(lo, hi, value=med, step=(hi - lo) / 200, label=f))
        sonuc = gr.Markdown()
        for k in kaydiricilar:
            k.change(sensor_tahmini, kaydiricilar, sonuc)
        demo.load(sensor_tahmini, kaydiricilar, sonuc)

demo.launch(share=True, debug=True)